In [1]:
import torch
import clip

In [2]:
# Set device
if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

device

'mps'

In [3]:
# Load the model and image preprocessing pipeline
model, preprocess = clip.load("ViT-B/32", device=device)


model.eval()  # Set to evaluation mode
print(f"Model loaded successfully on {device}")

/Users/karlxu/workspace/github/xuyuji9000/machine-learning-playground/01_models/01_clip/01_try-with-openai-library/venv/lib/python3.14/site-packages/torch/jit/_serialization.py:170: FutureWarning: `torch.jit.load` is not supported in Python 3.14+ and may break. Please switch to `torch.export`.
  warnings.warn(


Model loaded successfully on mps


In [4]:
from PIL import Image

# Load the image and preprocess it
image_path = "02-man-without-hat.png"
image = preprocess(Image.open(image_path)).unsqueeze(0).to(device)

# Calculate the embedding using CLIP's encode_image
with torch.no_grad():
    image_embedding = model.encode_image(image)

# If you want to inspect the embedding:
print("Embedding shape:", image_embedding.shape)
print("Embedding (first 5 values):", image_embedding[0][:5])

Embedding shape: torch.Size([1, 512])
Embedding (first 5 values): tensor([-0.0584, -0.2661, -0.1969,  0.0774, -0.2612], device='mps:0',
       dtype=torch.float16)


In [5]:
clothing_list = [
    "Hat", "T-shirt", "Blouse", "Button-down Shirt", "Polo Shirt", "Tank Top", 
    "Hoodie", "Sweater", "Cardigan", "Tunic", "Jeans", 
    "Trousers", "Slacks", "Chinos", "Shorts", "Skirt", 
    "Leggings", "Culottes", "Joggers", "Dress", "Jumpsuit", 
    "Suit", "Blazer", "Trench Coat", "Overcoat", "Parka", 
    "Denim Jacket", "Bomber Jacket", "Windbreaker", "Gown", "Pajamas", 
    "Nightgown", "Robe", "Bikini", "Swim Trunks", "Socks", 
    "Tights", "Beanie", "Baseball Cap", "Beret", "Fedora"
]

# Calculate text embeddings for the clothing list using CLIP's tokenizer and encode_text
with torch.no_grad():
    text_tokens = clip.tokenize(clothing_list).to(device)
    text_embeddings = model.encode_text(text_tokens)

print("Text embeddings shape:", text_embeddings.shape)
print("Text embedding for first item (first 5 values):", text_embeddings[0][:5])



Text embeddings shape: torch.Size([41, 512])
Text embedding for first item (first 5 values): tensor([ 0.0667,  0.1737, -0.3616, -0.0557, -0.0345], device='mps:0',
       dtype=torch.float16)


In [6]:
# Calculate cosine similarities between the image embedding and the text embeddings
import torch.nn.functional as F

# Normalize the embeddings
image_embedding_norm = F.normalize(image_embedding, p=2, dim=-1)
text_embeddings_norm = F.normalize(text_embeddings, p=2, dim=-1)

# Compute cosine similarity
similarities = (image_embedding_norm @ text_embeddings_norm.T).squeeze(0)

# Print top 5 most similar clothing items
top5_indices = similarities.topk(5).indices
print("Top 5 most similar clothing items:")
for idx in top5_indices:
    print(f"{clothing_list[idx]} (score: {similarities[idx].item():.4f})")

Top 5 most similar clothing items:
Button-down Shirt (score: 0.2247)
Bomber Jacket (score: 0.2111)
Overcoat (score: 0.2107)
Polo Shirt (score: 0.2095)
Cardigan (score: 0.2068)
